In [ ]:
# TODO: ipynb analyzer doesn't work if import and API call is in different cells,
# as the alias is not persistent across different cells (each cell as a .py file)

In [ ]:
# Python and ipynb API analyzer

import ast
import json
from collections import defaultdict
from pathlib import Path
from tempfile import NamedTemporaryFile


def _annotate_parents(tree: ast.AST) -> None:
    """Add `.parent` links so we can walk up the tree."""
    for parent in ast.walk(tree):
        for child in ast.iter_child_nodes(parent):
            child.parent = parent


def _in_type_checking_block(node: ast.AST) -> bool:
    """Check if in an `if TYPE_CHECKING:` block."""
    while hasattr(node, "parent"):
        parent = node.parent
        if isinstance(parent, ast.If):
            if isinstance(parent.test, ast.Name) and parent.test.id == "TYPE_CHECKING":
                return True
        node = parent
    return False


class ApiAnalyzerPy(ast.NodeVisitor):
    """API analyzer for .py file."""

    def __init__(self, package: str) -> None:
        self.package = package

        # alias map: local name → full path
        self.aliases: dict[str, str] = {}
        # usage counts
        self.usage: dict[str, int] = defaultdict(int)

    def visit_Import(self, node) -> None:
        if _in_type_checking_block(node):
            return
        for alias in node.names:
            if alias.name.startswith(self.package):
                asname = alias.asname or alias.name.split(".")[-1]
                self.aliases[asname] = alias.name

    def visit_ImportFrom(self, node) -> None:
        if _in_type_checking_block(node):
            return
        if node.module and node.module.startswith(self.package):
            for alias in node.names:
                asname = alias.asname or alias.name
                self.aliases[asname] = f"{node.module}.{alias.name}"

    def visit_Call(self, node) -> None:
        """Track function/method calls."""
        if isinstance(node.func, ast.Attribute):
            base = node.func.value
            if isinstance(base, ast.Name) and base.id in self.aliases:
                full = f"{self.aliases[base.id]}.{node.func.attr}"
                self.usage[full] += 1
        self.generic_visit(node)


def analyze_py(path: str | Path, package: str) -> tuple[dict[str, str], dict[str, int]]:
    """
    Analyze a Python file for package API usage.

    Returns:
        aliases (dict): Mapping of local names → full package paths.
        usage (dict): Mapping of package API calls → count.
    """
    path = Path(path)

    if path.suffix != ".py":
        raise ValueError(f"cannot analyze non-py file: {path}")

    text = path.read_text(encoding="utf-8")

    tree = ast.parse(text)

    _annotate_parents(tree)

    analyzer = ApiAnalyzerPy(package)
    analyzer.visit(tree)
    return analyzer.aliases, dict(analyzer.usage)


def analyze_notebook(
    path: str | Path, package: str
) -> tuple[dict[str, str], dict[str, int]]:
    def clean_notebook_code(code: str) -> str:
        cleaned: list[str] = []
        for line in code.splitlines():
            if not line or line.startswith(("!", "%", "?")):
                continue  # skip shell/magic/help commands
            cleaned.append(line)
        return "\n".join(cleaned)

    path = Path(path)
    if path.suffix != ".ipynb":
        raise ValueError(f"cannot analyze non-ipynb file: {path}")

    nb = json.loads(path.read_text(encoding="utf-8"))
    aliases, usage = {}, {}

    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        code = "".join(cell.get("source", []))
        code = clean_notebook_code(code)
        if not code.strip():
            continue

        # Write code to a temporary .py file
        with NamedTemporaryFile("w", suffix=".py", delete=False) as tmp:
            tmp.write(code)
            tmp_path = Path(tmp.name)

        _alias, _usage = analyze_py(tmp_path, package)

        # Merge results
        aliases.update(_alias)
        for key, val in _usage.items():
            usage[key] = usage.get(key, 0) + val

        tmp_path.unlink(missing_ok=False)

    return aliases, usage

In [ ]:
"""Sanity check with .py and .ipynb files."""
test_py_file = "./dependent-repos/pymatviz/pymatviz/chem_env.py"
aliases, usage = analyze_py(test_py_file, "pymatgen")
print(aliases)
print(usage)

test_ipynb_file = "./dependent-repos/pymatviz/examples/widgets/jupyter_demo.ipynb"
aliases, usage = analyze_notebook(test_ipynb_file, "pymatgen")
print(aliases)
print(usage)

{'local_env': 'pymatgen.analysis.local_env'}
{'pymatgen.analysis.local_env.LocalStructOrderParams': 1, 'pymatgen.analysis.local_env.CrystalNN': 1}
{'Composition': 'pymatgen.core.Composition', 'Lattice': 'pymatgen.core.Lattice', 'Structure': 'pymatgen.core.Structure'}
{'pymatgen.core.Lattice.cubic': 1}


In [ ]:
def analyze_paths(
    paths: str | Path | list[str | Path],
    package: str,
    exclude: str | list[str] | None = None,
) -> tuple[dict[str, str], dict[str, int]]:
    """
    Analyze Python (.py) and Jupyter (.ipynb) files for API usage of `package`.

    Args:
        paths: a single file/dir path, or a list of them
        package: the package to analyze (e.g. "pymatgen")
        exclude: optional subdir name(s) to exclude, relative to each path
            e.g. exclude=["tests", "docs", ".venv"]

    Returns:
        aliases (dict): merged local → full package paths
        usage (dict): merged API usage counts across all files
    """
    if isinstance(paths, (str, Path)):
        paths = [paths]
    paths = [Path(p) for p in paths]

    if exclude is None:
        exclude = []
    elif isinstance(exclude, str):
        exclude = [exclude]
    exclude = set(exclude)

    all_aliases: dict[str, str] = {}
    all_usage: dict[str, int] = defaultdict(int)

    def should_skip(p: Path) -> bool:
        # skip hidden files/dirs
        if p.name.startswith("."):
            return True
        # skip if any parent directory matches an excluded subdir
        if any(parent.name in exclude for parent in p.parents):
            return True
        return False

    for path in paths:
        if should_skip(path):
            continue

        if path.is_dir():
            candidates = path.rglob("*")
        else:
            candidates = [path]

        for f in candidates:
            if should_skip(f):
                continue
            if f.suffix == ".py":
                aliases, usage = analyze_py(f, package)
            elif f.suffix == ".ipynb":
                try:
                    aliases, usage = analyze_notebook(f, package)
                except Exception as e:
                    print(f"⚠️ Skipping {f} (error: {e})")
                    continue
            else:
                continue

            all_aliases.update(aliases)
            for k, v in usage.items():
                all_usage[k] += v

    return all_aliases, dict(all_usage)

In [ ]:
# Test `analyze_paths` on path | list[path]
test_dir = "./dependent-repos/pymatviz/"
aliases_single, usage_single = analyze_paths(test_dir, "pymatgen")

aliases_list, usage_list = analyze_paths([test_dir], "pymatgen")
assert aliases_single == aliases_list
assert usage_single == usage_list
print(aliases_list)
print(usage_list)

# Test exclusion
aliases_empty, usage_empty = analyze_paths(
    test_dir, "pymatgen", exclude=["examples", "pymatviz", "tests"]
)
assert not aliases_empty, aliases_empty
assert not usage_empty, usage_empty

{'Lattice': 'pymatgen.core.Lattice', 'Structure': 'pymatgen.core.Structure', 'AseAtomsAdaptor': 'pymatgen.io.ase.AseAtomsAdaptor', 'PhononBands': 'pymatgen.phonon.bandstructure.PhononBandStructureSymmLine', 'PhononDos': 'pymatgen.phonon.dos.PhononDos', 'Composition': 'pymatgen.core.Composition', 'IStructure': 'pymatgen.core.IStructure', 'DiffractionPattern': 'pymatgen.analysis.diffraction.xrd.DiffractionPattern', 'XRDCalculator': 'pymatgen.analysis.diffraction.xrd.XRDCalculator', 'MPRester': 'pymatgen.ext.matproj.MPRester', 'local_env': 'pymatgen.analysis.local_env', 'SiteCollection': 'pymatgen.core.SiteCollection', 'get_pmg_structure': 'pymatgen.io.phonopy.get_pmg_structure', 'MSONAtoms': 'pymatgen.io.ase.MSONAtoms', 'CrystalNN': 'pymatgen.analysis.local_env.CrystalNN', 'NearNeighbors': 'pymatgen.analysis.local_env.NearNeighbors', 'VoronoiNN': 'pymatgen.analysis.local_env.VoronoiNN', 'Species': 'pymatgen.core.periodic_table.Species', 'PeriodicSite': 'pymatgen.core.PeriodicSite', 'Elem

In [ ]:
# Clone top X repos
import pandas as pd
import subprocess
from pathlib import Path

INPUT_FILE: str = "pymatgen_dependents_with_stars.csv"
OUTPUT_DIR: Path = Path("dependent-repos")
TOP_N: int = 5  # TODO: try with 5 first

OUTPUT_DIR.mkdir(exist_ok=True)

repos_df: pd.DataFrame = pd.read_csv(INPUT_FILE).head(TOP_N)

for _, row in repos_df.iterrows():
    repo_url: str = row["repo_url"]
    repo_name: str = Path(repo_url).name
    target_path: str = OUTPUT_DIR / repo_name

    print(f"Processing {repo_name}...")

    try:
        subprocess.run(
            ["git", "clone", "--depth=1", repo_url],
            cwd=OUTPUT_DIR,
            check=True,
            capture_output=True,
            text=True,
        )
        print(f"✓ Cloned {repo_name}")
    except subprocess.CalledProcessError as e:
        # git clone returns 128 if the folder exists, but we ignore it
        if "already exists" in e.stderr:
            print(f"⚠ Skipping {repo_name}, already cloned")
        else:
            print(f"✗ Failed cloning {repo_name}: {e.stderr.strip()}")

Processing materials_discovery...
⚠ Skipping materials_discovery, already cloned
Processing Uni-Mol...
⚠ Skipping Uni-Mol, already cloned
Processing AIRS...
⚠ Skipping AIRS, already cloned
Processing mat2vec...
⚠ Skipping mat2vec, already cloned
Processing megnet...
⚠ Skipping megnet, already cloned


In [ ]:
# Analyze top X repos

from dataclasses import dataclass
from pathlib import Path


@dataclass
class RepoConfig:
    exclude: list[str] | None = None


# Need to manually build a glob mapping for each repo
REPO_PATH_MAP: dict[str, RepoConfig] = {
    "materials_discovery": RepoConfig(exclude=["tests", "docs"]),
    "Uni-Mol": RepoConfig(exclude=["tests"]),
    # "AIRS": RepoConfig(exclude=[]),
    "mat2vec": RepoConfig(exclude=[]),
    "megnet": RepoConfig(exclude=["tests", "examples"]),
}


def analyze_all_repos(base_dir: str | Path, package: str):
    results = {}
    for repo_name, repo_config in REPO_PATH_MAP.items():
        print(f"\n🔎 Analyzing {repo_name}...")
        aliases, usage = analyze_paths(
            f"{base_dir}/{repo_name}", package, repo_config.exclude
        )
        results[repo_name] = {"aliases": aliases, "usage": usage}
        print(f"  → Found {len(aliases)} aliases, {len(usage)} API usages")
    return results


results = analyze_all_repos(OUTPUT_DIR, "pymatgen")

for repo, result in results.items():
    print(f"{repo=}")
    print(f"{result["aliases"]=}")
    print(f"{result["usage"]=}")
    print("-" * 20, end="\n\n")


🔎 Analyzing materials_discovery...
  → Found 7 aliases, 0 API usages

🔎 Analyzing Uni-Mol...
⚠️ Skipping dependent-repos/Uni-Mol/unimol_docking_v2/interface/posebuster_demo.ipynb (error: unexpected indent (<unknown>, line 2))


<unknown>:13: SyntaxWarning: invalid escape sequence '\W'
<unknown>:6: SyntaxWarning: invalid escape sequence '\W'


⚠️ Skipping dependent-repos/Uni-Mol/unimol/notebooks/unimol_mol_property_demo.ipynb (error: unexpected indent (<unknown>, line 21))
⚠️ Skipping dependent-repos/Uni-Mol/unimol/notebooks/unimol_posebuster_demo.ipynb (error: unexpected indent (<unknown>, line 10))
⚠️ Skipping dependent-repos/Uni-Mol/unimol/notebooks/unimol_pocket_repr_demo.ipynb (error: invalid syntax (<unknown>, line 3))
⚠️ Skipping dependent-repos/Uni-Mol/unimol/notebooks/unimol_binding_pose_demo.ipynb (error: invalid syntax (<unknown>, line 6))
⚠️ Skipping dependent-repos/Uni-Mol/unimol/notebooks/unimol_mol_repr_demo.ipynb (error: unexpected indent (<unknown>, line 2))
  → Found 0 aliases, 0 API usages

🔎 Analyzing mat2vec...
  → Found 3 aliases, 0 API usages

🔎 Analyzing megnet...
⚠️ Skipping dependent-repos/megnet/multifidelity/codes_for_plots/Figure4/plot_figure4_scatter_error.ipynb (error: invalid syntax (<unknown>, line 1))


<unknown>:8: SyntaxWarning: invalid escape sequence '\C'
<unknown>:55: SyntaxWarning: invalid escape sequence '\d'
<unknown>:43: SyntaxWarning: invalid escape sequence '\m'
<unknown>:23: SyntaxWarning: invalid escape sequence '\m'
<unknown>:25: SyntaxWarning: invalid escape sequence '\m'
<unknown>:36: SyntaxWarning: invalid escape sequence '\m'
<unknown>:37: SyntaxWarning: invalid escape sequence '\D'
<unknown>:47: SyntaxWarning: invalid escape sequence '\m'
<unknown>:23: SyntaxWarning: invalid escape sequence '\m'
<unknown>:28: SyntaxWarning: invalid escape sequence '\m'
<unknown>:35: SyntaxWarning: invalid escape sequence '\m'
<unknown>:36: SyntaxWarning: invalid escape sequence '\D'
<unknown>:74: SyntaxWarning: invalid escape sequence '\D'
<unknown>:72: SyntaxWarning: invalid escape sequence '\D'
<unknown>:11: SyntaxWarning: invalid escape sequence '\d'


  → Found 23 aliases, 6 API usages
repo='materials_discovery'
result["aliases"]={'mg': 'pymatgen', 'ComputedEntry': 'pymatgen.entries.computed_entries.ComputedEntry', 'phase_diagram': 'pymatgen.analysis.phase_diagram', 'interface_reactions': 'pymatgen.analysis.interface_reactions', 'Structure': 'pymatgen.core.Structure', 'StructureMatcher': 'pymatgen.analysis.structure_matcher.StructureMatcher', 'cif': 'pymatgen.io.cif'}
result["usage"]={}
--------------------

repo='Uni-Mol'
result["aliases"]={}
result["usage"]={}
--------------------

repo='mat2vec'
result["aliases"]={'Element': 'pymatgen.core.periodic_table.Element', 'Composition': 'pymatgen.core.composition.Composition', 'CompositionError': 'pymatgen.core.composition.CompositionError'}
result["usage"]={}
--------------------

repo='megnet'
result["aliases"]={'Structure': 'pymatgen.core.Structure', 'MPRester': 'pymatgen.MPRester', 'Lattice': 'pymatgen.core.Lattice', 'Molecule': 'pymatgen.core.Molecule', 'find_points_in_spheres': 'py